# Climate-Driven Vector-Borne Dengue Outbreak Surveillance Model
### **Project CCHAIN** | *Climate Change, Health, and Artificial Intelligence in the Philippines*

This notebook demonstrates an end-to-end data pipeline linking **ERA5 climate reanalysis**, **DOH/LGU epidemiological surveillance records**, and **Google Open Buildings / WorldPop built-environment data** across all 80 barangays in **Cagayan de Oro City (2003–2022)**.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Set paths
DATA_DIR = Path('./data/cchain_raw')
OUTPUT_DIR = Path('./data/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PILOT_CITY_CODE = 'PH104305000'  # Cagayan de Oro City
TARGET_DISEASE = 'DENGUE FEVER'
print(f'Pipeline configured for {PILOT_CITY_CODE} - Target: {TARGET_DISEASE}')

## 1. Ingest Real Geographic Boundaries & Barangay Master Table

In [ ]:
df_loc = pd.read_csv(DATA_DIR / 'location.csv')
cdo_brgys = df_loc[df_loc['adm3_pcode'] == PILOT_CITY_CODE][
    ['adm1_en', 'adm2_en', 'adm3_pcode', 'adm3_en', 'adm4_pcode', 'adm4_en', 'brgy_total_area']
].drop_duplicates().reset_index(drop=True)

target_pcodes = cdo_brgys['adm4_pcode'].unique()
print(f'Identified {len(target_pcodes)} barangays in Cagayan de Oro.')
cdo_brgys.head()

## 2. Ingest & Aggregate Disease Surveillance (LGU Dengue Records)

In [ ]:
df_lgu = pd.read_csv(DATA_DIR / 'disease_lgu_disaggregated_totals.csv')

# Filter for Cagayan de Oro and Dengue Fever
df_dengue = df_lgu[
    (df_lgu['adm3_pcode'] == PILOT_CITY_CODE) &
    (df_lgu['disease_common_name'] == TARGET_DISEASE) &
    (df_lgu['adm4_pcode'].isin(target_pcodes))
].copy()

# Standardize monthly timestamp
df_dengue['date'] = pd.to_datetime(df_dengue['date']).dt.to_period('M').dt.to_timestamp()

# Aggregate across demographics
df_health_agg = df_dengue.groupby(['adm4_pcode', 'date'], as_index=False).agg(
    dengue_cases=('case_total', 'sum'),
    dengue_deaths=('death_total', 'sum')
)
print(f'Aggregated {len(df_health_agg)} barangay-month surveillance observations.')
df_health_agg.head()

## 3. Ingest & Aggregate Atmospheric Data (ERA5 Reanalysis)

In [ ]:
df_clim = pd.read_csv(DATA_DIR / 'climate_atmosphere.csv')
df_clim = df_clim[df_clim['adm4_pcode'].isin(target_pcodes)].copy()
df_clim['date'] = pd.to_datetime(df_clim['date']).dt.to_period('M').dt.to_timestamp()

# Aggregate daily climate data to monthly statistics
df_clim_agg = df_clim.groupby(['adm4_pcode', 'date'], as_index=False).agg(
    pr_monthly_total_mm=('pr', 'sum'),
    tave_monthly_mean_c=('tave', 'mean'),
    tmin_monthly_mean_c=('tmin', 'mean'),
    tmax_monthly_mean_c=('tmax', 'mean'),
    heat_index_monthly_mean_c=('heat_index', 'mean'),
    heat_index_monthly_max_c=('heat_index', 'max'),
    rh_monthly_mean_pct=('rh', 'mean'),
    wind_speed_monthly_mean=('wind_speed', 'mean'),
    solar_rad_monthly_mean=('solar_rad', 'mean')
)
print(f'Aggregated {len(df_clim_agg)} monthly climate vectors.')
df_clim_agg.head()

## 4. Ingest Built Environment & Demographics (Google Open Buildings & WorldPop)

In [ ]:
df_bldgs = pd.read_csv(DATA_DIR / 'google_open_buildings.csv')
df_bldgs_cdo = df_bldgs[df_bldgs['adm4_pcode'].isin(target_pcodes)][
    ['adm4_pcode', 'google_bldgs_count', 'google_bldgs_density', 'google_bldgs_pct_built_up_area', 'google_bldgs_area_mean']
].drop_duplicates(subset=['adm4_pcode'])

df_pop = pd.read_csv(DATA_DIR / 'worldpop_population.csv')
df_pop_cdo = df_pop[df_pop['adm4_pcode'].isin(target_pcodes)].sort_values(by='date').groupby('adm4_pcode').last().reset_index()[
    ['adm4_pcode', 'pop_count_total', 'pop_density_mean']
]

df_static = df_bldgs_cdo.merge(df_pop_cdo, on='adm4_pcode', how='left')
df_static.head()

## 5. Space-Time Alignment Grid & Biological Lag Feature Engineering

In [ ]:
min_date = df_clim_agg['date'].min()
max_date = df_clim_agg['date'].max()
all_dates = pd.date_range(start=min_date, end=max_date, freq='MS')

# Create full grid
grid_idx = pd.MultiIndex.from_product([target_pcodes, all_dates], names=['adm4_pcode', 'date']).to_frame().reset_index(drop=True)

# Relational merge
df_merged = grid_idx.merge(cdo_brgys, on='adm4_pcode', how='left')
df_merged = df_merged.merge(df_clim_agg, on=['adm4_pcode', 'date'], how='left')
df_merged = df_merged.merge(df_health_agg, on=['adm4_pcode', 'date'], how='left')
df_merged = df_merged.merge(df_static, on='adm4_pcode', how='left')

# Clean missing cases
df_merged['dengue_cases'] = df_merged['dengue_cases'].fillna(0).astype(int)
df_merged['dengue_deaths'] = df_merged['dengue_deaths'].fillna(0).astype(int)
df_merged = df_merged.sort_values(by=['adm4_pcode', 'date']).reset_index(drop=True)

# Engineer 1-month, 2-month, and 3-month weather lag features
for lag in [1, 2, 3]:
    df_merged[f'pr_total_mm_lag_{lag}m'] = df_merged.groupby('adm4_pcode')['pr_monthly_total_mm'].shift(lag)
    df_merged[f'heat_index_mean_lag_{lag}m'] = df_merged.groupby('adm4_pcode')['heat_index_monthly_mean_c'].shift(lag)
    df_merged[f'tave_mean_lag_{lag}m'] = df_merged.groupby('adm4_pcode')['tave_monthly_mean_c'].shift(lag)

df_merged['pr_rolling_3m_avg'] = df_merged.groupby('adm4_pcode')['pr_monthly_total_mm'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)

# Outbreak Definition: 75th percentile of historical caseload per barangay
df_merged['brgy_p75_threshold'] = df_merged.groupby('adm4_pcode')['dengue_cases'].transform(
    lambda x: max(5, x.quantile(0.75))
)
df_merged['is_outbreak'] = (df_merged['dengue_cases'] >= df_merged['brgy_p75_threshold']).astype(int)

df_final = df_merged.dropna(subset=['pr_total_mm_lag_3m']).copy()
output_csv = OUTPUT_DIR / 'cchain_cdo_dengue_surveillance_ready.csv'
df_final.to_csv(output_csv, index=False)

print(f'Exported {df_final.shape[0]} rows x {df_final.shape[1]} features to {output_csv}')
df_final[['adm4_en', 'date', 'dengue_cases', 'pr_monthly_total_mm', 'pr_total_mm_lag_1m', 'heat_index_mean_lag_2m', 'is_outbreak']].head(10)

## 6. Machine Learning Outbreak Surveillance Model (Random Forest)

In [ ]:
feature_cols = [
    'pr_monthly_total_mm', 'tave_monthly_mean_c', 'heat_index_monthly_mean_c',
    'heat_index_monthly_max_c', 'rh_monthly_mean_pct', 'wind_speed_monthly_mean',
    'pr_total_mm_lag_1m', 'pr_total_mm_lag_2m', 'pr_total_mm_lag_3m',
    'heat_index_mean_lag_1m', 'heat_index_mean_lag_2m', 'heat_index_mean_lag_3m',
    'tave_mean_lag_1m', 'pr_rolling_3m_avg',
    'google_bldgs_density', 'google_bldgs_pct_built_up_area', 'pop_density_mean'
]

X = df_final[feature_cols].fillna(0)
y = df_final['is_outbreak']

# Temporal split (Train: 2003-2018, Test: 2019-2022)
train_mask = df_final['date'] < '2019-01-01'
test_mask = df_final['date'] >= '2019-01-01'

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

rf = RandomForestClassifier(n_estimators=150, max_depth=7, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print(f'Test Outbreak ROC-AUC Score: {roc_auc_score(y_test, y_prob):.3f}')
print('\nClassification Report:\n', classification_report(y_test, y_pred))

# Feature Importances
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print('\nFeature Importances:')
display(importances.to_frame('Relative Importance'))